# Detection

Point prediction — cell / nucleus centroids — on a frozen foundation-model
token grid:

```
Dataset(+points) -> DenseTileFeatureExtractor -> train (decoder + head) -> evaluate
```

The encoder emits a **token grid per tile**, and a **decoder** smooths it into
a per-class peak heatmap the head reads points back from.
[Segmentation](walkthrough-segmentation.ipynb) is the same dense flow with mask
supervision and a per-pixel head.

> Tiny synthetic data, CPU-only, ungated encoder — the numbers are
> meaningless; the point is the API. We use
> [phikon](https://huggingface.co/owkin/phikon) at its native **224 px**
> window (a 14×14 token grid), which avoids position-embedding interpolation.

## ⚠️ Scaffolding (not soma API)

Dense supervision lives in per-sample files, not a scalar `label`: `dataset.csv`
carries `sample_id, image_path, points_path`, where the points file is a CSV of
`x, y, class` in ROI-pixel coordinates. We fabricate small **224 px ROI tiles**
(the dense flow consumes fixed-size tiles/ROIs, not whole WSIs) plus their
point files.

In [ ]:
import logging, warnings
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)

import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile

WORK = Path(tempfile.mkdtemp(prefix='soma-detection-'))
ROIS = WORK / 'rois'; POINTS = WORK / 'points'
for d in (ROIS, POINTS): d.mkdir()
rng = np.random.default_rng(0)

SIZE = 224          # phikon native window
SPACING = 0.5       # microns/pixel
NUM_CLASSES = 3     # 0 = background, 1, 2 = cell classes

def make_roi(path):
    img = np.clip(np.stack([np.full((SIZE, SIZE), 150),
                            np.full((SIZE, SIZE), 70),
                            np.full((SIZE, SIZE), 160)], -1).astype(np.int16)
                  + rng.integers(-30, 30, (SIZE, SIZE, 3)), 0, 255).astype(np.uint8)
    tifffile.imwrite(path, img, photometric='rgb', tile=(SIZE, SIZE),
                     resolution=(20000, 20000), resolutionunit='CENTIMETER')

def make_points(path):
    # a few cells per class, in ROI-pixel coordinates
    pts = [(56, 56, 0), (112, 112, 1), (160, 160, 1)]
    pd.DataFrame(pts, columns=['x', 'y', 'class']).to_csv(path, index=False)

ids = [f'roi{i:02d}' for i in range(8)]
for sid in ids:
    make_roi(ROIS / f'{sid}.tif')
    make_points(POINTS / f'{sid}.csv')

split = ['train'] * 4 + ['tune'] * 2 + ['test'] * 2
splits_csv = WORK / 'splits.csv'
pd.DataFrame({'sample_id': ids, 'split': split, 'fold': 0}).to_csv(splits_csv, index=False)

img_paths = [str(ROIS / f'{s}.tif') for s in ids]

# Feature extraction only needs the images; supervision lives in the point manifest.
extract_csv = WORK / 'extract.csv'
pd.DataFrame({'sample_id': ids, 'image_path': img_paths, 'label': 0}).to_csv(extract_csv, index=False)

det_csv = WORK / 'det.csv'
pd.DataFrame({'sample_id': ids, 'image_path': img_paths,
              'points_path': [str(POINTS / f'{s}.csv') for s in ids]}).to_csv(det_csv, index=False)
print(pd.read_csv(det_csv).head(3).to_string(index=False))

## 1. Extract dense token grids

`DenseTileFeatureExtractor` reads each ROI at `spacing_um` and runs the frozen
encoder to produce a `(feature_dim, gh, gw)` grid per sample, stored in a
`DenseFeatureStore`. With phikon at 224 px and patch-16 that's a 14×14 grid.

In [ ]:
from soma import (
    Dataset, DenseTileFeatureExtractor, EncoderConfig, CacheConfig,
)

extractor = DenseTileFeatureExtractor(
    Dataset(extract_csv),
    EncoderConfig(name='phikon'),
    target_size=SIZE,
    spacing_um=SPACING,
    backend='openslide',
    cache=CacheConfig(enabled=False),
)
dense_store = extractor.run(str(WORK / 'dense'))
print('dense grids for', len(dense_store.available_samples), 'ROIs')

## 2. Train the decoder + head

`TaskConfig('detection')` renders each annotated point as a peak Gaussian; the
decoder smooths the grid into a peak heatmap, and the head recovers points
(local-maxima + NMS) scored with **F1 at a matching distance δ**.
`match_distance` and `sigma` are given in **µm**. `DetectionManifest` is the
dense counterpart of `Dataset`.

In [ ]:
from soma.dataset import DetectionManifest
from soma import (
    Splits, DecoderConfig, TaskConfig, TrainingConfig, EvalConfig,
    PreprocessingConfig, train,
)

det_manifest = DetectionManifest(det_csv)
det_splits = Splits(splits_csv, det_manifest)

det_result = train(
    feature_store=dense_store,
    dataset=det_manifest,
    splits=det_splits,
    dataset_type='detection',
    decoder=DecoderConfig(name='lightweight_conv'),
    task=TaskConfig(name='detection', params={
        'num_classes': NUM_CLASSES,
        'match_distance': 2.0,   # microns
        'sigma': 0.7,            # microns
    }),
    training=TrainingConfig(epochs=3, batch_size=2, learning_rate=1e-3, seed=0),
    evaluation=EvalConfig(metrics=['mean_f1', 'f1_per_class']),
    preprocessing=PreprocessingConfig(requested_spacing_um=SPACING, requested_tile_size_px=SIZE),
    run_dir=str(WORK / 'runs' / 'detection'),
)
print('detection run dir:', det_result.run_dir)

## 3. The one-shot `Pipeline` equivalent

`Pipeline` collapses extract + train + evaluate into a single config-driven
call.

*(Shown for reference, not executed.)*

```python
from soma import (
    Pipeline, PipelineConfig, PreprocessingConfig, EncoderConfig,
    DecoderConfig, TaskConfig, TrainingConfig, EvalConfig, CacheConfig,
)

config = PipelineConfig(
    dataset_csv=str(det_csv),
    splits_csv=str(splits_csv),
    output_root='output/detection',
    dataset_type='detection',
    preprocessing=PreprocessingConfig(
        backend='openslide', requested_tile_size_px=224, requested_spacing_um=0.5,
    ),
    encoder=EncoderConfig(name='phikon'),
    decoder=DecoderConfig(name='lightweight_conv'),
    task=TaskConfig(name='detection', params={'num_classes': 3, 'match_distance': 2.0, 'sigma': 0.7}),
    training=TrainingConfig(epochs=3, batch_size=2, learning_rate=1e-3),
    evaluation=EvalConfig(metrics=['mean_f1', 'f1_per_class']),
    cache=CacheConfig(enabled=True),
)
results = Pipeline(config).run()
```